In [4]:
import gymnasium as gym
from h1 import h1
import numpy as np
import torch
import humanoid_bench

In [5]:
env = gym.make(
        "h1-maze-v0",
        render_mode="rgb_array",
)
env.reset()
data = env.unwrapped.named.data

joint_positions = h1.fk_joint_positions(h1.body_tree["pelvis"], data.qpos)

print(joint_positions)

{'free_base': array([0.00411148, 0.00274744, 0.97407004]), 'left_hip_yaw': array([0.00734147, 0.09173127, 0.80065339]), 'left_hip_roll': array([0.0468034 , 0.0914078 , 0.80126569]), 'left_hip_pitch': array([0.04773424, 0.20676016, 0.80221326]), 'left_knee': array([0.21149411, 0.20843651, 0.43727506]), 'left_ankle': array([0.06144009, 0.21269305, 0.06651142]), 'right_hip_yaw': array([ 0.00631087, -0.08325916,  0.79914173]), 'right_hip_roll': array([ 0.04577085, -0.08376904,  0.7997524 ]), 'right_hip_pitch': array([ 0.04429313, -0.19911665,  0.79893096]), 'right_knee': array([ 0.20428358, -0.1985555 ,  0.43232117]), 'right_ankle': array([ 0.05534228, -0.19400387,  0.06111264]), 'torso': array([0.00411148, 0.00274744, 0.97407004]), 'left_shoulder_pitch': array([0.00366396, 0.15438797, 1.40541703]), 'left_shoulder_roll': array([-0.0013948 ,  0.21105382,  1.38935031]), 'left_shoulder_yaw': array([-3.45120386e-05,  2.13017349e-01,  1.25507155e+00]), 'left_elbow': array([0.02046955, 0.2157757

In [6]:
print(data.xanchor)

FieldIndexer(xanchor):
                          x         y         z         
 0            free_base [ 0.00411   0.00275   0.974   ]
 1         left_hip_yaw [ 0.00734   0.0917    0.801   ]
 2        left_hip_roll [ 0.0468    0.0914    0.801   ]
 3       left_hip_pitch [ 0.0477    0.207     0.802   ]
 4            left_knee [ 0.211     0.208     0.437   ]
 5           left_ankle [ 0.0614    0.213     0.0665  ]
 6        right_hip_yaw [ 0.00631  -0.0833    0.799   ]
 7       right_hip_roll [ 0.0458   -0.0838    0.8     ]
 8      right_hip_pitch [ 0.0443   -0.199     0.799   ]
 9           right_knee [ 0.204    -0.199     0.432   ]
10          right_ankle [ 0.0553   -0.194     0.0611  ]
11                torso [ 0.00411   0.00275   0.974   ]
12  left_shoulder_pitch [ 0.00366   0.154     1.41    ]
13   left_shoulder_roll [-0.00178   0.213     1.41    ]
14    left_shoulder_yaw [-0.000355  0.214     1.28    ]
15           left_elbow [ 0.0202    0.217     1.08    ]
16 right_shoulder_pitch 

In [7]:
# Build mapping from joint order to xanchor indices (assumed order)
joint_to_xanchor_mapping = {
    "free_base": 0,
    "left_hip_yaw": 1,
    "left_hip_roll": 2,
    "left_hip_pitch": 3,
    "left_knee": 4,
    "left_ankle": 5,
    "right_hip_yaw": 6,
    "right_hip_roll": 7,
    "right_hip_pitch": 8,
    "right_knee": 9,
    "right_ankle": 10,
    "torso": 11,
    "left_shoulder_pitch": 12,
    "left_shoulder_roll": 13,
    "left_shoulder_yaw": 14,
    "left_elbow": 15,
    "right_shoulder_pitch": 16,
    "right_shoulder_roll": 17,
    "right_shoulder_yaw": 18,
    "right_elbow": 19,
}

print(f"{'Joint':20s} | {'FK':30s} | {'Xanchor':30s} | {'Max Diff':8s}")
print("-" * 100)

max_diff_overall = 0.0
max_diff_joint = ""
sum_err = 0.0
count = 0

for joint_name, xanchor_idx in joint_to_xanchor_mapping.items():
    if joint_name in joint_positions:
        fk_pos = joint_positions[joint_name]
        xanchor_pos = data.xanchor[xanchor_idx]
        diff = np.abs(fk_pos - xanchor_pos)
        max_diff = np.max(diff)
        rmse = np.sqrt(np.mean((fk_pos - xanchor_pos) ** 2))
        
        sum_err += rmse
        count += 1
        
        if max_diff > max_diff_overall:
            max_diff_overall = max_diff
            max_diff_joint = joint_name
        
        fk_str = np.array2string(fk_pos, precision=3, separator=',', suppress_small=True)
        xanchor_str = np.array2string(xanchor_pos, precision=3, separator=',', suppress_small=True)
        
        print(f"{joint_name:20s} | {fk_str:30s} | {xanchor_str:30s} | {max_diff:8.6f}")

avg_rmse = (sum_err / max(count,1))
print(f"\nLargest difference: {max_diff_overall:.6f} at joint '{max_diff_joint}'")
print(f"Average RMSE: {avg_rmse:.6f}")


Joint                | FK                             | Xanchor                        | Max Diff
----------------------------------------------------------------------------------------------------
free_base            | [0.004,0.003,0.974]            | [0.004,0.003,0.974]            | 0.000000
left_hip_yaw         | [0.007,0.092,0.801]            | [0.007,0.092,0.801]            | 0.000000
left_hip_roll        | [0.047,0.091,0.801]            | [0.047,0.091,0.801]            | 0.000000
left_hip_pitch       | [0.048,0.207,0.802]            | [0.048,0.207,0.802]            | 0.000000
left_knee            | [0.211,0.208,0.437]            | [0.211,0.208,0.437]            | 0.000000
left_ankle           | [0.061,0.213,0.067]            | [0.061,0.213,0.067]            | 0.000000
right_hip_yaw        | [ 0.006,-0.083, 0.799]         | [ 0.006,-0.083, 0.799]         | 0.000000
right_hip_roll       | [ 0.046,-0.084, 0.8  ]         | [ 0.046,-0.084, 0.8  ]         | 0.000000
right_hip_pitch  